In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI
import json
import time
import tiktoken

print("正在初始化测评环境...")

helper_model_path = "/root/autodl-tmp/Qwen/Qwen3___5-9B"
helper_tokenizer = AutoTokenizer.from_pretrained(helper_model_path)
helper_model = AutoModelForCausalLM.from_pretrained(
    helper_model_path, torch_dtype=torch.bfloat16, device_map="auto"
)

DEEPSEEK_API_KEY = ""
deepseek_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text):
    return len(enc.encode(text))

print("模型加载完毕！")

class ComparativeEvaluator:
    def __init__(self, chunk_size=20, keep_turns=20): 
        self.chunk_size = chunk_size
        self.keep_turns = keep_turns
        
        self.exp_history = []
        self.exp_json_summary = "{}"
        
        self.baseline_history = []
        
        self.evaluation_results = []
        
    def run_single_turn(self, turn_id, speaker_role, user_input):
        print(f"\n" + "-"*50)
        print(f"🎬 [第 {turn_id} 轮] 角色: {speaker_role}")
        print(f"💬 内容: {user_input[:40]}...")
        
        formatted_input = f"[{speaker_role}]说：{user_input}"
        
        turn_metrics = {
            "turn_id": turn_id,
            "speaker": speaker_role,
            "original_text": user_input,
            "baseline": {},
            "experimental": {}
        }
        
        self.baseline_history.append({"role": "user", "content": formatted_input})
        
        baseline_system = "你是一个沉浸式的角色扮演大模型。请根据当前剧情发展，扮演合适的角色或给出合理的推进回复。"
        baseline_messages = [{"role": "system", "content": baseline_system}] + self.baseline_history
        
        turn_metrics["baseline"]["input_tokens"] = count_tokens(json.dumps(baseline_messages, ensure_ascii=False))
        
        start_time = time.time()
        try:
            res_base = deepseek_client.chat.completions.create(
                model="deepseek-chat", messages=baseline_messages, temperature=0.7
            )
            baseline_reply = res_base.choices[0].message.content
        except Exception as e:
            baseline_reply = f"Error: {e}"
            
        turn_metrics["baseline"]["latency_sec"] = round(time.time() - start_time, 2)
        turn_metrics["baseline"]["reply"] = baseline_reply
        self.baseline_history.append({"role": "assistant", "content": baseline_reply})

        self.exp_history.append({"role": "user", "content": formatted_input})
        helper_latency = 0
        helper_triggered = False
        
        if len(self.exp_history) >= (self.chunk_size + self.keep_turns):
            helper_triggered = True
            slice_index = len(self.exp_history) - self.keep_turns
            chunk_to_compress = self.exp_history[:slice_index]
            self.exp_history = self.exp_history[slice_index:]
            
            latest_dialogue = "\n".join([t['content'] for t in chunk_to_compress if t['role'] == 'user'])
            
            helper_input_text = f"【历史提纲】:\n{self.exp_json_summary}\n\n【最新对话】:\n{latest_dialogue}"
            
            system_prompt = "你是一个长对话记忆辅助模型。请根据【历史提纲】和【最新对话】，提取并更新为精简的状态提纲，重点保留对当前剧情有影响的核心伏笔和最新进展（输出JSON格式）。 {\"enable_thinking\": false}"
            messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": helper_input_text}]
            
            text = helper_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            model_inputs = helper_tokenizer([text], return_tensors="pt").to(helper_model.device)
            
            h_start = time.time()
            generated_ids = helper_model.generate(**model_inputs, max_new_tokens=1024, do_sample=False, temperature=None)
            generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
            self.exp_json_summary = helper_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
            helper_latency = time.time() - h_start
            print(f"   ⚡ 实验组触发记忆压缩! (Helper耗时: {helper_latency:.2f}s)")
            
        exp_system = f"你是一个沉浸式的角色扮演大模型。\n以下是你当前必须记住的核心背景提纲：\n{self.exp_json_summary}\n请严格遵循设定回复。"
        exp_messages = [{"role": "system", "content": exp_system}] + self.exp_history
        
        turn_metrics["experimental"]["input_tokens"] = count_tokens(json.dumps(exp_messages, ensure_ascii=False))
        
        start_time = time.time()
        try:
            res_exp = deepseek_client.chat.completions.create(
                model="deepseek-chat", messages=exp_messages, temperature=0.7
            )
            exp_reply = res_exp.choices[0].message.content
        except Exception as e:
            exp_reply = f"Error: {e}"
            
        turn_metrics["experimental"]["deepseek_latency_sec"] = round(time.time() - start_time, 2)
        turn_metrics["experimental"]["helper_latency_sec"] = round(helper_latency, 2)
        turn_metrics["experimental"]["total_latency_sec"] = round(turn_metrics["experimental"]["deepseek_latency_sec"] + helper_latency, 2)
        turn_metrics["experimental"]["helper_triggered"] = helper_triggered
        turn_metrics["experimental"]["reply"] = exp_reply
        
        extracted_foreshadows = []
        display_summary = self.exp_json_summary
        
        try:
            clean_str = self.exp_json_summary.strip()
            if clean_str.startswith("```json"):
                clean_str = clean_str[7:]
            if clean_str.endswith("```"):
                clean_str = clean_str[:-3]
                
            parsed_json = json.loads(clean_str.strip())
            
            extracted_foreshadows = parsed_json.pop("key_items_or_events", [])
            
            display_summary = json.dumps(parsed_json, ensure_ascii=False, indent=2)
            
        except Exception:
            extracted_foreshadows = ["(提纲解析中，暂未提取到结构化伏笔)"]
            
        turn_metrics["experimental"]["current_json_summary"] = display_summary
        turn_metrics["experimental"]["tracked_foreshadows"] = extracted_foreshadows

        self.exp_history.append({"role": "assistant", "content": exp_reply})
        
        self.evaluation_results.append(turn_metrics)
        print(f"   Input Token 消耗对比 -> 原生组: {turn_metrics['baseline']['input_tokens']} | 实验组: {turn_metrics['experimental']['input_tokens']}")
        return turn_metrics

    def save_results(self, filename="./comparative_eval_results(5)_v2.json"):
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(self.evaluation_results, f, ensure_ascii=False, indent=2)
        print(f"\n完整测评数据（含对话与指标）已保存至: {filename}")


print("\n🚀 开始读取剧本并执行 A/B 双轨测评...")

with open("test_dataset(5).json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

if isinstance(test_data, list):
    test_data = test_data[0]  

dialogue_list = test_data.get("dialogue", [])
print(f"成功读取剧本主题: {test_data.get('theme', '未知')}")
print(f"共计 {len(dialogue_list)} 轮对话将参与测试。\n")

evaluator = ComparativeEvaluator(chunk_size=60, keep_turns=10)

for i, turn in enumerate(dialogue_list):
    role = turn.get("role", "未知角色")
    content = turn.get("content", "")
    
    evaluator.run_single_turn(turn_id=i+1, speaker_role=role, user_input=content)
    
    time.sleep(1)

evaluator.save_results()
print("剧本测评完成！")

🔄 正在初始化测评环境...


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

✅ 模型加载完毕！

🚀 开始读取剧本并执行 A/B 双轨测评...
📚 成功读取剧本主题: 末日废土：收音机的频率
📏 共计 432 轮对话将参与测试。


--------------------------------------------------
🎬 [第 1 轮] 角色: 旁白
💬 内容: 废弃的高速公路上，两辆生锈的汽车横在路中央，阳光透过破碎的云层洒在满是裂缝的柏油...
   📊 Input Token 消耗对比 -> 原生组: 236 | 实验组: 242

--------------------------------------------------
🎬 [第 2 轮] 角色: 老狗
💬 内容: （头也不抬，语气冷淡）别浪费时间翻那些没用的东西，重点找水和弹药。...
   📊 Input Token 消耗对比 -> 原生组: 403 | 实验组: 365

--------------------------------------------------
🎬 [第 3 轮] 角色: 石头
💬 内容: （趴在后座，手在座椅缝隙里摸索）我知道，队长。但万一有急救包之类的呢？...
   📊 Input Token 消耗对比 -> 原生组: 573 | 实验组: 519

--------------------------------------------------
🎬 [第 4 轮] 角色: 老狗
💬 内容: （嗤笑一声）急救包？这破车连轮胎都没了，你就是把座椅拆了也找不到半卷绷带。...
   📊 Input Token 消耗对比 -> 原生组: 762 | 实验组: 685

--------------------------------------------------
🎬 [第 5 轮] 角色: 石头
💬 内容: （手碰到一个硬物，眼睛一亮）等等……我摸到个东西。...
   📊 Input Token 消耗对比 -> 原生组: 918 | 实验组: 817

--------------------------------------------------
🎬 [第 6 轮] 角色: 老狗
💬 内容: （站起身，警惕地扫视四周）别一惊一乍的，是老鼠还是蟑螂？...
   📊 Input Token 消耗对比 ->

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


   ⚡ 实验组触发记忆压缩! (Helper耗时: 17.29s)
   📊 Input Token 消耗对比 -> 原生组: 7347 | 实验组: 1641

--------------------------------------------------
🎬 [第 37 轮] 角色: 老狗
💬 内容: （突然停下，举起手示意石头停下）等等……前面有动静。...
   📊 Input Token 消耗对比 -> 原生组: 7506 | 实验组: 1830

--------------------------------------------------
🎬 [第 38 轮] 角色: 石头
💬 内容: （立刻蹲下，压低声音）丧尸？...
   📊 Input Token 消耗对比 -> 原生组: 7682 | 实验组: 1960

--------------------------------------------------
🎬 [第 39 轮] 角色: 老狗
💬 内容: （眯着眼看远处）不确定……好像是个人影。...
   📊 Input Token 消耗对比 -> 原生组: 7882 | 实验组: 2133

--------------------------------------------------
🎬 [第 40 轮] 角色: 石头
💬 内容: （紧张地握紧手中的铁管）活人还是死人？...
   📊 Input Token 消耗对比 -> 原生组: 8081 | 实验组: 2256

--------------------------------------------------
🎬 [第 41 轮] 角色: 老狗
💬 内容: （缓缓拔出腰间的刀）活人不会在中午站在路中间不动。...
   📊 Input Token 消耗对比 -> 原生组: 8266 | 实验组: 2458

--------------------------------------------------
🎬 [第 42 轮] 角色: 石头
💬 内容: （咽了口唾沫）那我们绕路？...
   📊 Input Token 消耗对比 -> 原生组: 8456 | 实验组: 2577

--------------------------------------

...

   ⚡ 实验组触发记忆压缩! (Helper耗时: 23.00s)
   📊 Input Token 消耗对比 -> 原生组: 79695 | 实验组: 3987

--------------------------------------------------
🎬 [第 367 轮] 角色: 老狗
💬 内容: （喘息着，抹了一把脸上的雨水）妈的，这群畜生跟上来了。塔基的焊门撑不了多久，最多...
   📊 Input Token 消耗对比 -> 原生组: 79908 | 实验组: 4449

--------------------------------------------------
🎬 [第 368 轮] 角色: 石头
💬 内容: （紧握着手中的撬棍，声音发抖）那……那我们怎么办？跳下去就是死路一条。...
   📊 Input Token 消耗对比 -> 原生组: 80170 | 实验组: 5034

--------------------------------------------------
🎬 [第 369 轮] 角色: 老狗
💬 内容: （环视操作室，目光扫过布满灰尘的控制台）别慌，这里有个基站发射台。我见过这玩意儿...
   📊 Input Token 消耗对比 -> 原生组: 80458 | 实验组: 5519

--------------------------------------------------
🎬 [第 370 轮] 角色: 石头
💬 内容: （眼睛一亮）驱散？怎么弄？...
   📊 Input Token 消耗对比 -> 原生组: 80615 | 实验组: 6002

--------------------------------------------------
🎬 [第 371 轮] 角色: 老狗
💬 内容: （快步走到控制台前，拍掉灰尘）高频声波。发射出去能干扰它们的听觉系统，让它们误以...
   📊 Input Token 消耗对比 -> 原生组: 80912 | 实验组: 6567

--------------------------------------------------
🎬 [第 372 轮] 角色: 石头
💬 内容: （凑过去看）那赶紧的啊！哪个按钮？...
   📊 I

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


   ⚡ 实验组触发记忆压缩! (Helper耗时: 20.56s)
   📊 Input Token 消耗对比 -> 原生组: 86695 | 实验组: 3208

--------------------------------------------------
🎬 [第 397 轮] 角色: 老狗
💬 内容: （缓缓转动旋钮，发出嘎吱声）现在呢？...
   📊 Input Token 消耗对比 -> 原生组: 86973 | 实验组: 3564

--------------------------------------------------
🎬 [第 398 轮] 角色: 石头
💬 内容: （摇头）还差一点，指针在104.3左右。...
   📊 Input Token 消耗对比 -> 原生组: 87241 | 实验组: 3915

--------------------------------------------------
🎬 [第 399 轮] 角色: 老狗
💬 内容: （继续拧，额头冒出冷汗）现在？...
   📊 Input Token 消耗对比 -> 原生组: 87442 | 实验组: 4423

--------------------------------------------------
🎬 [第 400 轮] 角色: 石头
💬 内容: （屏住呼吸）再往右一点点……对，就那儿！停！...
   📊 Input Token 消耗对比 -> 原生组: 87703 | 实验组: 4807

--------------------------------------------------
🎬 [第 401 轮] 角色: 老狗
💬 内容: （松开手，看向窗外）好，准备发射。你退后，捂住耳朵。...
   📊 Input Token 消耗对比 -> 原生组: 88035 | 实验组: 5458

--------------------------------------------------
🎬 [第 402 轮] 角色: 石头
💬 内容: （后退两步，双手捂住耳朵）你自己也捂啊！...
   📊 Input Token 消耗对比 -> 原生组: 88327 | 实验组: 6553

---------------------

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


   ⚡ 实验组触发记忆压缩! (Helper耗时: 19.76s)
   📊 Input Token 消耗对比 -> 原生组: 94511 | 实验组: 3418

--------------------------------------------------
🎬 [第 427 轮] 角色: 老狗
💬 内容: （点头）前提是电池够用。先下去检查一下铁门，看还能不能修。...
   📊 Input Token 消耗对比 -> 原生组: 94718 | 实验组: 3896

--------------------------------------------------
🎬 [第 428 轮] 角色: 石头
💬 内容: （站起来，拍了拍身上的灰）好。对了，那个频率……我们要记下来吗？...
   📊 Input Token 消耗对比 -> 原生组: 95000 | 实验组: 4237

--------------------------------------------------
🎬 [第 429 轮] 角色: 老狗
💬 内容: （从口袋里掏出一支快没水的笔，在墙上歪歪扭扭写下“104.5”）记在这儿。以后这...
   📊 Input Token 消耗对比 -> 原生组: 95251 | 实验组: 4690

--------------------------------------------------
🎬 [第 430 轮] 角色: 石头
💬 内容: （看着墙上的数字，低声说）104.5……收音机里的幽灵信号。...
   📊 Input Token 消耗对比 -> 原生组: 95466 | 实验组: 5024

--------------------------------------------------
🎬 [第 431 轮] 角色: 老狗
💬 内容: （拍拍他的肩膀）别叫它幽灵了。叫它救命信号。...
   📊 Input Token 消耗对比 -> 原生组: 95682 | 实验组: 5431

--------------------------------------------------
🎬 [第 432 轮] 角色: 旁白
💬 内容: 两人沿着螺旋楼梯缓缓走下，脚步声在空旷的塔内回荡。塔外的晨光渐渐明亮，废墟中一片..